In [ ]:
import logging
import sys
import torch.distributed as dist
from collections.abc import MutableMapping
from logging import getLogger

from ray import tune

from recbole.config import Config
from recbole.data import (
    create_dataset,
    data_preparation,
)
from recbole.data.transform import construct_transform
from recbole.utils import (
    init_logger,
    get_model,
    get_trainer,
    init_seed,
    set_color,
    get_flops,
    get_environment,
)
import torch
import os


In [2]:
dataset_name = "Amazon_Instruments"
if dataset_name == 'Amazon_Arts':
    model_path = '/data1/CoAuthor1/Py_projects/datasets/Amazon_Arts/DMF-Jul-03-2024_16-11-39.pth'
elif dataset_name == "Amazon_Games":
    model_path = '/data1/CoAuthor1/Py_projects/datasets/Amazon_Games/DMF-Jul-03-2024_16-13-31.pth'
elif dataset_name == "Amazon_Instruments":
    model_path = '/data1/CoAuthor1/Py_projects/datasets/Amazon_Instruments/DMF-Jul-03-2024_16-15-08.pth'
# model_path = '/data1/CoAuthor1/Py_projects/datasets/Amazon_Instruments/DMF-Jul-03-2024_16-15-08.pth'
# model_path = '/data1/CoAuthor1/Py_projects/datasets/Amazon_Games/DMF-Jul-03-2024_16-13-31.pth'
data_path = f'/data1/CoAuthor1/Py_projects/datasets/{dataset_name}'



In [3]:
def load_data_and_model(model_file,data_path=''):
    r"""Load filtered dataset, split dataloaders and saved model.

    Args:
        model_file (str): The path of saved model file.

    Returns:
        tuple:
            - config (Config): An instance object of Config, which record parameter information in :attr:`model_file`.
            - model (AbstractRecommender): The model load from :attr:`model_file`.
            - dataset (Dataset): The filtered dataset.
            - train_data (AbstractDataLoader): The dataloader for training.
            - valid_data (AbstractDataLoader): The dataloader for validation.
            - test_data (AbstractDataLoader): The dataloader for testing.
    """
    import torch

    checkpoint = torch.load(model_file)
    config = checkpoint["config"]
    if data_path:
        config['data_path'] = data_path
    init_seed(config["seed"], config["reproducibility"])
    init_logger(config)
    logger = getLogger()
    logger.info(config)

    dataset = create_dataset(config)
    logger.info(dataset)
    train_data, valid_data, test_data = data_preparation(config, dataset)

    init_seed(config["seed"], config["reproducibility"])
    model = get_model(config["model"])(config, train_data._dataset).to(config["device"])
    model.load_state_dict(checkpoint["state_dict"])
    model.load_other_parameter(checkpoint.get("other_parameter"))

    return config, model, dataset, train_data, valid_data, test_data

In [ ]:

os.environ["CUDA_VISIBLE_DEVICES"] = "5"
config, model, dataset, train_data, valid_data, test_data  = load_data_and_model(
    model_file=model_path,
    data_path=data_path
)
model

In [5]:
user_id = torch.tensor([1])
user_embedding = model.get_user_embedding(user=user_id)
user_embedding

tensor([[-0.2050, -0.3228, -0.4647,  0.0741,  0.2104, -0.0580,  0.3864,  0.0679,
          0.2985,  1.1948, -0.6040,  0.8059, -0.0105,  0.0624, -0.4696, -0.4778,
         -0.1234,  0.1707,  0.1359, -0.6502, -1.0439, -0.1461, -0.0099, -0.1653,
          0.0108, -0.0570,  0.2514, -0.6372,  0.1007,  0.4549,  0.2030, -0.8342,
          0.1831,  0.4379,  0.3588,  0.2486, -0.2859,  0.4303,  0.0322,  0.0834,
         -0.5527,  0.4949, -0.2245,  0.4269,  0.5653, -0.5955,  0.0624, -0.4397,
         -0.5758,  0.8753,  0.5611,  0.5607,  0.3894,  0.0542, -0.7078,  0.0908,
          0.2956,  0.0521,  1.2198,  0.0805, -0.0507,  0.2376,  0.2854,  0.5165]],
       device='cuda:0', grad_fn=<MmBackward0>)

In [ ]:
item_embeddings = model.get_item_embedding()

In [11]:
dataset.field2token_id['item_id']

{'[PAD]': 0,
 '0739079891': 1,
 '1480360295': 2,
 '1928571018': 3,
 '9792372326': 4,
 'B00000J50W': 5,
 'B00001W0DH': 6,
 'B00001W0DT': 7,
 'B00004TT3S': 8,
 'B00004UE29': 9,
 'B00004UE2D': 10,
 'B00004Y2V2': 11,
 'B00004Y2V1': 12,
 'B00004Y2UT': 13,
 'B00005ML71': 14,
 'B000068NSX': 15,
 'B000068NTC': 16,
 'B000068NTU': 17,
 'B000068NVJ': 18,
 'B000068NYM': 19,
 'B000068NUQ': 20,
 'B000068NZG': 21,
 'B000068NZD': 22,
 'B000068NZB': 23,
 'B000068NSS': 24,
 'B000068NYU': 25,
 'B000068O29': 26,
 'B000068NW9': 27,
 'B000068NW8': 28,
 'B000068NYP': 29,
 'B000068NUX': 30,
 'B000068NUW': 31,
 'B000068O58': 32,
 'B000068O35': 33,
 'B000068O4F': 34,
 'B000068OAY': 35,
 'B000068O59': 36,
 'B000068OHN': 37,
 'B000068OEW': 38,
 'B000068O3X': 39,
 'B000068OAT': 40,
 'B000068O2P': 41,
 'B00006HO3R': 42,
 'B00006HO3L': 43,
 'B00006HOLL': 44,
 'B00006I51S': 45,
 'B00006I5SD': 46,
 'B00006I51V': 47,
 'B00006I5R7': 48,
 'B00006I5SC': 49,
 'B00006I523': 50,
 'B00006I5SA': 51,
 'B00006J04Z': 52,
 'B00006

In [12]:
item_embeddings[dataset.field2token_id['item_id']['0739079891']]

tensor([1.1205, 1.5896, 0.0000, 0.0000, 0.0000, 1.2421, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.9660, 1.7664, 0.0000, 0.0000,
        1.6825, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0759, 0.3391,
        0.0000, 1.1143, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0212,
        0.0000, 1.1792, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.8767, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        1.2301, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000], device='cuda:0', grad_fn=<SelectBackward0>)

In [63]:
from recbole.data import interaction
dic_inter = {'user_id' : torch.tensor([1,]), "item_id" : torch.tensor([10,])}
new_inter = interaction.Interaction(dic_inter)
model.predict(new_inter)

tensor([0.5000], device='cuda:0', grad_fn=<SigmoidBackward0>)

In [64]:
import numpy as np
arr = model.get_user_embedding(user=user_id).cpu().detach().numpy().reshape(-1)
arr

array([-0.20496945, -0.3227965 , -0.46465656,  0.07411462,  0.21042001,
       -0.05802272,  0.3863871 ,  0.06786796,  0.29852992,  1.1947727 ,
       -0.6040243 ,  0.80586547, -0.01047385,  0.06238733, -0.46958777,
       -0.47777006, -0.12338277,  0.17069095,  0.13585356, -0.65016896,
       -1.0438855 , -0.14611039, -0.00989997, -0.1652616 ,  0.01077699,
       -0.05702282,  0.2513708 , -0.6372216 ,  0.10067737,  0.45489034,
        0.20301446, -0.8342236 ,  0.1831184 ,  0.43794265,  0.35882545,
        0.24858814, -0.28585914,  0.43025333,  0.03221573,  0.08335116,
       -0.5527386 ,  0.49488503, -0.22450387,  0.426907  ,  0.56526303,
       -0.595502  ,  0.06238782, -0.43970704, -0.5758101 ,  0.8753104 ,
        0.56106377,  0.5607401 ,  0.3893988 ,  0.05418219, -0.70784473,
        0.0908366 ,  0.29562366,  0.05212505,  1.2197778 ,  0.08054218,
       -0.05067191,  0.23756938,  0.28542727,  0.51651657], dtype=float32)

In [65]:
len(dataset.field2id_token['user_id'])

27404

In [66]:
dataset.dataset_name

'Amazon_Instruments'

In [67]:
# from sklearn.cluster import KMeans
# # load user_id from the dataset
# user_id = dataset.field2id_token['user_id']

# # cluster user_id from model's user embeddings
# uid_range = len(user_id)

# total_user_emb = []

# # ignore padding user
# for i in range(1, uid_range):
#     user_emb = model.get_user_embedding(torch.tensor([i, ])).cpu().detach().numpy().reshape(-1)
#     total_user_emb.append(user_emb)

# # use KMeans to cluster user embeddings
# n_clusters = 100
# random_seed = 2024
# kmeans = KMeans(n_clusters=n_clusters, random_state=random_seed).fit(total_user_emb)
# user_cluster = kmeans.labels_

In [68]:
# user_cluster = user_cluster.tolist()

In [69]:
# user_cluster

In [70]:
# use leave_one_out to get the dataset
datasets = dataset.leave_one_out(group_by='user_id', leave_one_mode='valid_and_test')
train_dataset, valid_dataset, test_dataset = datasets[0], datasets[1], datasets[2]

In [71]:
# get user's history interactions, rating and length (train_dataset as example)
train_history_matrix, train_history_value, train_length = train_dataset.history_item_matrix('timestamp')
valid_history_matrix, valid_history_value, valid_length = valid_dataset.history_item_matrix('timestamp')
test_history_matrix, test_history_value, test_length = test_dataset.history_item_matrix('timestamp')

In [72]:
for i in range(train_history_matrix.shape[0]):
    arr = train_history_value[i]
    # sort indices with reversed sequence
    sorted_arr = torch.argsort(arr, descending=False)
    train_history_matrix[i] = train_history_matrix[i][sorted_arr]
    train_history_value[i] = train_history_value[i][sorted_arr]
    # move zero elements to the end
    zero_idx = torch.where(train_history_matrix[i] == 0)[0]
    non_zero_idx = torch.where(train_history_matrix[i] != 0)[0]
    train_history_matrix[i] = torch.cat([train_history_matrix[i][non_zero_idx], train_history_matrix[i][zero_idx]])
    train_history_value[i] = torch.cat([train_history_value[i][non_zero_idx], train_history_value[i][zero_idx]])

In [73]:
# get user's history interactions
train_history_matrix[1]

tensor([   1,  638, 1592, 1258, 5027, 1209,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,   

In [74]:
train_history_value[1]

tensor([1.4778e+09, 1.4778e+09, 1.4778e+09, 1.4778e+09, 1.4778e+09, 1.4778e+09,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+

In [75]:
# get user's history interaction length
# history_len[15403]

In [76]:
len(dataset.field2token_id['item_id'])

10450

In [77]:
len(train_dataset.field2token_id['item_id'])

10450

In [78]:
len(dataset.field2token_id['user_id'])

27404

In [79]:
len(train_dataset.field2token_id['user_id'])

27404

## get the interaction sequence of every user

In [80]:
# concat train, valid, test history matrix 

In [81]:
# train
train_item_re_id_inters_dict = {}
for uid_int in range(1, train_history_matrix.size(0)):
    train_item_re_id_inters_dict[uid_int] = train_history_matrix[uid_int].tolist()[:train_length[uid_int]]
train_item_re_id_inters_dict # user_id -> item_re_id

{1: [1, 638, 1592, 1258, 5027, 1209],
 2: [5882, 4663, 372, 349, 7384, 1, 4972],
 3: [8758, 1455, 6341],
 4: [6052, 1, 7902],
 5: [1, 1104, 8703, 4972],
 6: [1, 4972, 3135],
 7: [2332, 4001, 1, 5071, 3278, 3278, 7933, 8652, 8065, 5153, 6667],
 8: [1, 2953, 2953],
 9: [7384, 526, 1],
 10: [949, 645, 1],
 11: [3538, 266, 172, 1],
 12: [8805, 1, 1209, 1258, 638],
 13: [2795, 2795, 2219, 2, 2315, 1],
 14: [4537,
  778,
  4972,
  6829,
  4460,
  3435,
  4801,
  9793,
  1,
  747,
  6761,
  5476,
  1396,
  243,
  7313,
  2106,
  4684,
  5719,
  1613,
  793,
  1858,
  3616,
  4589,
  4953,
  4552,
  2874,
  2874,
  2584],
 15: [2874, 2874, 3995, 1],
 16: [5291, 585, 3110, 3110, 1, 3331],
 17: [3829, 2730, 4537],
 18: [7384, 1, 6177],
 19: [5627, 3964, 4972, 1, 1831, 5738],
 20: [3984, 1688, 6287, 2463, 2779, 2779, 1, 744],
 21: [4972, 5223, 1],
 22: [5389, 6216, 2451, 1, 5402],
 23: [146,
  4344,
  27,
  7763,
  7371,
  1597,
  1,
  1577,
  4632,
  9337,
  2779,
  2779,
  1837,
  1837],
 24: [

In [82]:
# valid
valid_item_re_id_inters_dict = {}
for uid_int in range(1, valid_history_matrix.size(0)):
    valid_item_re_id_inters_dict[uid_int] = valid_history_matrix[uid_int].tolist()[:valid_length[uid_int]]
# test
test_item_re_id_inters_dict = {}
for uid_int in range(1, test_history_matrix.size(0)):
    test_item_re_id_inters_dict[uid_int] = test_history_matrix[uid_int].tolist()[:test_length[uid_int]]

In [83]:
# concat train, valid, test
item_re_id_inters_dict = {}
for uid_int in train_item_re_id_inters_dict.keys():
    item_re_id_inters_dict[uid_int] = train_item_re_id_inters_dict[uid_int] + valid_item_re_id_inters_dict[uid_int] + test_item_re_id_inters_dict[uid_int]
item_re_id_inters_dict

{1: [1, 638, 1592, 1258, 5027, 1209, 5917, 9100],
 2: [5882, 4663, 372, 349, 7384, 1, 4972, 7902, 8241],
 3: [8758, 1455, 6341, 1, 4957],
 4: [6052, 1, 7902, 8424, 10369],
 5: [1, 1104, 8703, 4972, 1236, 1726],
 6: [1, 4972, 3135, 3135, 4096],
 7: [2332,
  4001,
  1,
  5071,
  3278,
  3278,
  7933,
  8652,
  8065,
  5153,
  6667,
  6659,
  7407],
 8: [1, 2953, 2953, 5291, 8362],
 9: [7384, 526, 1, 538, 966],
 10: [949, 645, 1, 4972, 5291],
 11: [3538, 266, 172, 1, 228, 4895],
 12: [8805, 1, 1209, 1258, 638, 1592, 3816],
 13: [2795, 2795, 2219, 2, 2315, 1, 4972, 4199],
 14: [4537,
  778,
  4972,
  6829,
  4460,
  3435,
  4801,
  9793,
  1,
  747,
  6761,
  5476,
  1396,
  243,
  7313,
  2106,
  4684,
  5719,
  1613,
  793,
  1858,
  3616,
  4589,
  4953,
  4552,
  2874,
  2874,
  2584,
  1077,
  10442],
 15: [2874, 2874, 3995, 1, 980, 5427],
 16: [5291, 585, 3110, 3110, 1, 3331, 4832, 8129],
 17: [3829, 2730, 4537, 1544, 1],
 18: [7384, 1, 6177, 9447, 9609],
 19: [5627, 3964, 4972, 1, 1

In [84]:

item_id_re2ori = {}
for key, value in dataset.field2token_id['item_id'].items():
    item_id_re2ori[value] = key
item_id_re2ori

{0: '[PAD]',
 1: '0739079891',
 2: '1480360295',
 3: '1928571018',
 4: '9792372326',
 5: 'B00000J50W',
 6: 'B00001W0DH',
 7: 'B00001W0DT',
 8: 'B00004TT3S',
 9: 'B00004UE29',
 10: 'B00004UE2D',
 11: 'B00004Y2V2',
 12: 'B00004Y2V1',
 13: 'B00004Y2UT',
 14: 'B00005ML71',
 15: 'B000068NSX',
 16: 'B000068NTC',
 17: 'B000068NTU',
 18: 'B000068NVJ',
 19: 'B000068NYM',
 20: 'B000068NUQ',
 21: 'B000068NZG',
 22: 'B000068NZD',
 23: 'B000068NZB',
 24: 'B000068NSS',
 25: 'B000068NYU',
 26: 'B000068O29',
 27: 'B000068NW9',
 28: 'B000068NW8',
 29: 'B000068NYP',
 30: 'B000068NUX',
 31: 'B000068NUW',
 32: 'B000068O58',
 33: 'B000068O35',
 34: 'B000068O4F',
 35: 'B000068OAY',
 36: 'B000068O59',
 37: 'B000068OHN',
 38: 'B000068OEW',
 39: 'B000068O3X',
 40: 'B000068OAT',
 41: 'B000068O2P',
 42: 'B00006HO3R',
 43: 'B00006HO3L',
 44: 'B00006HOLL',
 45: 'B00006I51S',
 46: 'B00006I5SD',
 47: 'B00006I51V',
 48: 'B00006I5R7',
 49: 'B00006I5SC',
 50: 'B00006I523',
 51: 'B00006I5SA',
 52: 'B00006J04Z',
 53: 'B0

In [85]:
# we need the form: user_id -> item_ori_id
item_ori_id_inters_dict = {}
for key, value in item_re_id_inters_dict.items():
    item_ori_id_inters_dict[key] = [item_id_re2ori[item_re_id] for item_re_id in value]
item_ori_id_inters_dict

{1: ['0739079891',
  'B0002E3CK4',
  'B0006LOBA8',
  'B0002H05BA',
  'B0051WAJ5S',
  'B0002H03YY',
  'B0094NVV5C',
  'B016W4O7BA'],
 2: ['B009115NQA',
  'B0047JP1D6',
  'B0002D09Y2',
  'B0002D0HXA',
  'B00IGUTZX4',
  '0739079891',
  'B004Z17008',
  'B00MXUJ394',
  'B00RGNR4TE'],
 3: ['B010CI6M88', 'B0002OOMU8', 'B00BL6JDUA', '0739079891', 'B004XNK7AI'],
 4: ['B009MBTCKW', '0739079891', 'B00MXUJ394', 'B00U49CFUA', 'B00YA48LJY'],
 5: ['0739079891',
  'B0002GLMEK',
  'B00YQXI2HU',
  'B004Z17008',
  'B0002H0A3S',
  'B0007Y09VO'],
 6: ['0739079891', 'B004Z17008', 'B0013MS0RE', 'B0013MS0RE', 'B0029RVWUO'],
 7: ['B000JUXKR6',
  'B002026DR0',
  '0739079891',
  'B0054QM4D6',
  'B0017SZ4OG',
  'B0017SZ4OG',
  'B00N4ZHPZ6',
  'B00XQFONF4',
  'B00OKA9F8O',
  'B005CXV6PI',
  'B00DMACALC',
  'B00DJU340G',
  'B00ILAQ974'],
 8: ['0739079891', 'B000YE8L7G', 'B000YE8L7G', 'B005MR6IHK', 'B00TB6WNQS'],
 9: ['B00IGUTZX4', 'B0002E1NNC', '0739079891', 'B0002E1NWI', 'B0002F7IEE'],
 10: ['B0002F79E8', 'B0002E3

In [86]:
dataset.dataset_name

'Amazon_Instruments'

In [87]:
import json

In [88]:
with open(f'/data1/CoAuthor1/Py_projects/MoRE/prepare_cluster_user/dataset/{dataset.dataset_name}/new_inters_dict.json','w') as f:
    json.dump(item_ori_id_inters_dict,f)

## data preprocessing

train

In [89]:
item_ori_id_set = set(dataset.field2token_id['item_id'].keys()) - set(['[PAD]'])
item_ori_id_list = list(item_ori_id_set)

In [90]:
from tqdm import tqdm
import random
random.seed(2024)

In [91]:
train_item_ori_id_inters_dict = {}
for uid in tqdm(item_ori_id_inters_dict.keys()):
    items = item_ori_id_inters_dict[uid][:-2]
    train_item_ori_id_inters_dict[uid] = {}
    for i in range(2, len(items)):
        # negative_item_set = item_ori_id_set - set(items[i])
        # candidate_set = [items[i]] + random.sample(list(negative_item_set), 49)
        candidate_set = [items[i]] + random.sample(item_ori_id_list, 49)
        random.shuffle(candidate_set)
        train_item_ori_id_inters_dict[uid][i] = {'inters' : items[:i], 
                                                      'target_item' : items[i], 
                                                      'candidate_set' : candidate_set}

100%|██████████| 27403/27403 [00:04<00:00, 6311.49it/s] 


In [92]:
with open(f'/data1/CoAuthor1/Py_projects/MoRE/prepare_cluster_user/dataset/{dataset.dataset_name}/new_train_samples.json','w')as f:
    json.dump(train_item_ori_id_inters_dict,f)

valid

In [93]:
valid_item_ori_id_inters_dict = {}
item_ori_id_list = list(item_ori_id_set)
for uid in tqdm(item_ori_id_inters_dict.keys()):
    items = item_ori_id_inters_dict[uid][:-1]
    valid_item_ori_id_inters_dict[uid] = {}
    i = -1
    # negative_item_set = item_ori_id_set - set(items[i])
    # candidate_set = [items[i]] + random.sample(list(negative_item_set), 49)
    candidate_set = [items[i]] + random.sample(item_ori_id_list, 49)
    random.shuffle(candidate_set)
    valid_item_ori_id_inters_dict[uid][i] = {'inters' : items[:i], 
                                                    'target_item' : items[i], 
                                                    'candidate_set' : candidate_set}

100%|██████████| 27403/27403 [00:00<00:00, 32541.02it/s]


In [94]:
with open(f'/data1/CoAuthor1/Py_projects/MoRE/prepare_cluster_user/dataset/{dataset.dataset_name}/new_valid_samples.json','w')as f:
    json.dump(valid_item_ori_id_inters_dict,f)

test

In [95]:
test_item_ori_id_inters_dict = {}
item_ori_id_list = list(item_ori_id_set)
for uid in tqdm(item_ori_id_inters_dict.keys()):
    items = item_ori_id_inters_dict[uid]
    test_item_ori_id_inters_dict[uid] = {}
    i = -1
    # negative_item_set = item_ori_id_set - set(items[i])
    # candidate_set = [items[i]] + random.sample(list(negative_item_set), 49)
    candidate_set = [items[i]] + random.sample(item_ori_id_list, 49)
    random.shuffle(candidate_set)
    test_item_ori_id_inters_dict[uid][i] = {'inters' : items[:i], 
                                                    'target_item' : items[i], 
                                                    'candidate_set' : candidate_set}

100%|██████████| 27403/27403 [00:00<00:00, 32713.92it/s]


In [96]:
with open(f'/data1/CoAuthor1/Py_projects/MoRE/prepare_cluster_user/dataset/{dataset.dataset_name}/new_test_samples.json','w')as f:
    json.dump(test_item_ori_id_inters_dict,f)